In [57]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import html as html_lib
import uuid

# =============================================================================
# 0. Common Settings
# =============================================================================
target_countries = ['Germany', 'France', 'United Kingdom', 'Czech Republic', 'Slovakia']
country_colors = {
    'Germany': '#1f77b4',
    'France': '#d62728',
    'United Kingdom': '#2ca02c',
    'Czech Republic': '#9467bd',
    'Slovakia': '#ff7f0e'
}
capitals = {
    'Germany': 'Berlin',
    'France': 'Paris',
    'United Kingdom': 'London',
    'Czech Republic': 'Prague',
    'Slovakia': 'Bratislava'
}

def gen_id():
    return str(uuid.uuid4())

# =============================================================================
# Plot 1: Spatial Distribution of Protests (Scatter Mapbox)
# =============================================================================
print("Generating Plot 1...")

df = pd.read_csv('Micro_With_VDem.csv', parse_dates=['event_date'])
df = df[df['country'].isin(target_countries)].copy()

df['Year'] = df['event_date'].dt.year
df['Quarter'] = df['event_date'].dt.quarter
df['Time_Quarter'] = df['Year'].astype(str) + '-Q' + df['Quarter'].astype(str)

df_agg = df.groupby(['country', 'location', 'latitude', 'longitude', 'Time_Quarter']).agg(
    total_protests=('event_id_cnty', 'count'),
    repressed_count=('sub_event_type', lambda x: x.isin([
        'Protest with intervention', 'Excessive force against protesters'
    ]).sum()),
    vdem_libdem=('v2x_libdem', 'mean'),
    vdem_rule=('v2x_rule', 'mean')
).reset_index()

df_map_filtered = df_agg[df_agg['total_protests'] >= 3].copy()

def classify_intervention(count):
    if count == 0:
        return '0: No Intervention'
    elif count <= 3:
        return '1-3: Low Intervention'
    else:
        return '4+: High Intervention'

df_map_filtered['Intervention_Level'] = df_map_filtered['repressed_count'].apply(classify_intervention)

color_map = {
    "0: No Intervention": "#AED6F1",
    "1-3: Low Intervention": "#F5B041",
    "4+: High Intervention": "#E74C3C"
}

quarters = sorted(df_map_filtered['Time_Quarter'].unique())
q0 = quarters[0]

fig1 = go.Figure()
df_q0_all = df_map_filtered[df_map_filtered['Time_Quarter'] == q0]

fig1.add_trace(go.Scattermapbox(
    lat=df_q0_all['latitude'],
    lon=df_q0_all['longitude'],
    mode='markers',
    name='All Countries',
    showlegend=False,
    marker=dict(
        size=df_q0_all['total_protests'] * 1.2,
        sizemin=4,
        color=df_q0_all['Intervention_Level'].map(color_map),
        opacity=0.85
    ),
    customdata=df_q0_all[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']],
    hovertemplate=(
        "<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
        "Total Protests: %{customdata[2]}<br>"
        "Interventions: %{customdata[3]}<br>"
        "<b>LibDem:</b> %{customdata[4]:.3f} | <b>Rule of Law:</b> %{customdata[5]:.3f}"
        "<extra></extra>"
    ),
    visible=True
))

for i, c in enumerate(target_countries):
    df_q0_c = df_q0_all[df_q0_all['country'] == c]
    fig1.add_trace(go.Scattermapbox(
        lat=df_q0_c['latitude'],
        lon=df_q0_c['longitude'],
        mode='markers',
        name=c,
        showlegend=False,
        marker=dict(
            size=df_q0_c['total_protests'] * 1.2,
            sizemin=4,
            color=df_q0_c['Intervention_Level'].map(color_map),
            opacity=0.85
        ),
        customdata=df_q0_c[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Total Protests: %{customdata[2]}<br>"
            "Interventions: %{customdata[3]}<br>"
            "<b>LibDem:</b> %{customdata[4]:.3f}"
            "<extra></extra>"
        ),
        visible=False
    ))

all_frames_1 = []
for q in quarters:
    df_q_all = df_map_filtered[df_map_filtered['Time_Quarter'] == q]
    frame_data = []
    frame_data.append(go.Scattermapbox(
        lat=df_q_all['latitude'],
        lon=df_q_all['longitude'],
        marker=dict(
            size=df_q_all['total_protests'] * 1.2,
            sizemin=4,
            color=df_q_all['Intervention_Level'].map(color_map)
        ),
        customdata=df_q_all[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']]
    ))
    for c in target_countries:
        df_q_c = df_q_all[df_q_all['country'] == c]
        frame_data.append(go.Scattermapbox(
            lat=df_q_c['latitude'],
            lon=df_q_c['longitude'],
            marker=dict(
                size=df_q_c['total_protests'] * 1.2,
                sizemin=4,
                color=df_q_c['Intervention_Level'].map(color_map)
            ),
            customdata=df_q_c[['location', 'country', 'total_protests', 'repressed_count', 'vdem_libdem', 'vdem_rule']]
        ))
    all_frames_1.append(go.Frame(data=frame_data, name=q))

fig1.frames = all_frames_1

country_buttons = [dict(label="All Countries", method="restyle", args=[{"visible": [True, False, False, False, False, False]}])]
for i in range(len(target_countries)):
    vis = [False] * 6
    vis[i + 1] = True
    country_buttons.append(dict(label=target_countries[i], method="restyle", args=[{"visible": vis}]))

custom_legend_title = (
    "<span style='font-size:18px;'><b>Spatial Distribution of Protests and State Repressive Intervention Hotspots in Five European Countries</b><br>"
    "<span style='display:inline-block;width:12px;height:12px;background-color:#666;border-radius:50%;margin-right:4px;'></span>"
    "<span style='color:#AED6F1; font-size:14px;'>No Intervention</span> &nbsp;&nbsp;&nbsp;&nbsp;"
    "<span style='display:inline-block;width:12px;height:12px;background-color:#666;border-radius:50%;margin-right:4px;'></span>"
    "<span style='color:#F5B041; font-size:14px;'>Low Intervention (1-3)</span> &nbsp;&nbsp;&nbsp;&nbsp;"
    "<span style='display:inline-block;width:12px;height:12px;background-color:#666;border-radius:50%;margin-right:4px;'></span>"
    "<span style='color:#E74C3C; font-size:14px;'>High Intervention (4+)</span> &nbsp;&nbsp;&nbsp;&nbsp;"
    "<span style='font-size:14px; color:#666;'>| Marker size indicates total protests; hover for V-Dem scores</span>"
    "</sup>"
)

fig1.update_layout(
    uirevision='locked',
    updatemenus=[
        dict(buttons=country_buttons, direction="down", x=0.0, y=1,
             xanchor="left", yanchor="top", showactive=True, bgcolor="white", bordercolor="#ccc"),
        dict(type="buttons", direction="left", x=0, y=-0.06, xanchor="left", yanchor="top",
             buttons=[
                 dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 800, "redraw": True}, "fromcurrent": True, "transition": {"duration": 300}}]),
                 dict(label="〓 Pause", method="animate", args=[[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}}])
             ],
             bgcolor="white", bordercolor="#ccc")
    ],
    sliders=[{"active": 0, "y": 0.0, "x": 0.15, "len": 0.85, "currentvalue": {"prefix": "Quarter: ", "font": {"size": 14, "color": "#0d6efd"}},
             "steps": [{"args": [[q], {"frame": {"duration": 500, "redraw": True}, "mode": "immediate", "transition": {"duration": 300}}], "label": q, "method": "animate"} for q in quarters]}],
    mapbox=dict(style="carto-positron", zoom=4, center=dict(lat=50, lon=10)),
    margin={"r": 0, "t": 100, "l": 0, "b": 0},
    height=720,
    title=custom_legend_title,
)


# =============================================================================
# Plot 2: Cross-National Repression Rate Trajectories
# =============================================================================
print("Generating Plot 2...")

df_panel = pd.read_csv('Final_Merged_Panel.csv')
df_panel['year_month'] = pd.to_datetime(df_panel['year_month'])
df_panel = df_panel[df_panel['country'].isin(target_countries)].copy()
df_panel = df_panel.sort_values(['country', 'year_month'])

rgba_map = {
    'Germany': 'rgba(31, 119, 180, 0.18)', 'France': 'rgba(214, 39, 40, 0.18)',
    'United Kingdom': 'rgba(44, 160, 44, 0.18)', 'Czech Republic': 'rgba(148, 103, 189, 0.18)',
    'Slovakia': 'rgba(255, 127, 14, 0.18)'
}

fig2 = go.Figure()

for country in target_countries:
    df_c = df_panel[df_panel['country'] == country]
    fig2.add_trace(go.Scatter(
        x=df_c['year_month'], y=df_c['crackdown_rate'], mode='lines', name=country,
        line=dict(width=2.5, color=country_colors[country]),
        hovertemplate="<b>%{text}</b><br>Date: %{x|%Y-%m}<br>Repression Rate: %{y:.3f}<extra></extra>",
        text=[country] * len(df_c), visible=True
    ))

df_avg = df_panel.groupby('year_month', as_index=False)['crackdown_rate'].mean()
fig2.add_trace(go.Scatter(
    x=df_avg['year_month'], y=df_avg['crackdown_rate'], mode='lines', name='Average',
    line=dict(width=3, color='black', dash='dash'),
    hovertemplate="<b>Overall Average</b><br>Date: %{x|%Y-%m}<br>Average Rate: %{y:.3f}<extra></extra>",
    visible=True
))

for country in target_countries:
    df_c = df_panel[df_panel['country'] == country]
    fig2.add_trace(go.Scatter(
        x=df_c['year_month'], y=df_c['crackdown_rate'], mode='lines', name=f"{country} filled",
        line=dict(width=3, color=country_colors[country]), fill='tozeroy', fillcolor=rgba_map[country],
        hovertemplate="<b>%{text}</b><br>Date: %{x|%Y-%m}<br>Repression Rate: %{y:.3f}<extra></extra>",
        text=[country] * len(df_c), visible=False
    ))

for country in target_countries:
    df_c = df_panel[df_panel['country'] == country].copy()
    peak_row = df_c.loc[df_c['crackdown_rate'].idxmax()]
    peak_label = f"{peak_row['year_month'].strftime('%Y-%m')}<br>{peak_row['crackdown_rate']:.2f}"
    fig2.add_trace(go.Scatter(
        x=[peak_row['year_month']], y=[peak_row['crackdown_rate']], mode='markers+text',
        name=f"{country} peak",
        marker=dict(size=10, color=country_colors[country], line=dict(color='white', width=1.5)),
        text=[peak_label], textposition='top center',
        hovertemplate=f"<b>{country} Peak</b><br>Date: {{%x|%Y-%m}}<br>Peak Rate: {{%y:.3f}}<extra></extra>",
        visible=False
    ))

buttons2 = []
visible_all = [True] * 6 + [False] * 10
buttons2.append(dict(label="All Countries", method="update", args=[{"visible": visible_all}, {"title": "<b>Cross-National Repression Rate Trajectories (2020-2025)</b><br><sup>Reveals variation in democratic resilience to dissent</sup>"}]))
visible_avg_only = [False] * 5 + [True] + [False] * 10
buttons2.append(dict(label="Average Only", method="update", args=[{"visible": visible_avg_only}, {"title": "<b>Five-Country Average Repression Rate Trajectory (2020-2025)</b><br><sup>Benchmark for individual country deviations</sup>"}]))
for i, country in enumerate(target_countries):
    visible_list = [False] * 16
    visible_list[5] = True
    visible_list[6 + i] = True
    visible_list[11 + i] = True
    buttons2.append(dict(label=country, method="update", args=[{"visible": visible_list}, {"title": f"<b>{country} Repression Rate Trajectory (2020-2025)</b><br><sup>Individual country dynamics vs. five-country average</sup>"}]))

fig2.update_layout(
    title="<span style='font-size:22px;'><b>Cross-National Repression Rate Trajectories (2020-2025)</b></span><br><br><span style='font-size:22px; color:#666;'><sup>Reveals variation in democratic resilience to dissent</sup></span>",
    height=580, plot_bgcolor="white", paper_bgcolor="white", legend_title="Country / Reference",
    margin=dict(l=70, r=40, t=120, b=70),
    updatemenus=[dict(buttons=buttons2, direction="down", showactive=True, x=1.02, xanchor="left", y=1.1, yanchor="top", bgcolor="white", bordercolor="lightgray", borderwidth=1, font=dict(size=12))]
)
fig2.update_xaxes(title_text="Date (Year-Month)", showline=True, linecolor='black', linewidth=1.2, showgrid=True, gridcolor='rgba(0,0,0,0.08)', ticks='outside', tickangle=45, tickformat="%Y-%m", dtick="M3", tickfont=dict(size=11), zeroline=False)
fig2.update_yaxes(title_text="Repression Rate", showline=True, linecolor='black', linewidth=1.2, showgrid=True, gridcolor='rgba(0,0,0,0.08)', ticks='outside', tickfont=dict(size=11), rangemode='tozero', zeroline=True, zerolinecolor='rgba(0,0,0,0.25)', zerolinewidth=1)


# =============================================================================
# Plot 3: Spatial Deterrence Comparison Map (Annual Aggregation)
# =============================================================================
print("Generating Plot 3...")

df_micro = pd.read_csv('Micro_With_VDem.csv')
df_micro_3 = df_micro[df_micro['country'].isin(target_countries)].copy()
df_micro_3['year'] = pd.to_datetime(df_micro_3['event_date']).dt.year
df_micro_3['is_intervened'] = df_micro_3['sub_event_type'].isin(['Protest with intervention', 'Excessive force against protesters']).astype(int)

country_colors_3 = {
    'Germany': '#1f77b4', 'France': '#17becf', 'United Kingdom': '#2ca02c',
    'Czech Republic': '#5C4033', 'Slovakia': '#ff7f0e'
}

valid_locs = df_micro_3.groupby(['country', 'location']).size().reset_index(name='total_count')
valid_locs = valid_locs[valid_locs['total_count'] >= 2][['country', 'location']]
df_filtered_3 = df_micro_3.merge(valid_locs, on=['country', 'location'], how='inner')

agg_df_3 = df_filtered_3.groupby(['country', 'location', 'latitude', 'longitude', 'year']).agg(
    total_events=('event_id_cnty', 'count'),
    intervened_events=('is_intervened', 'sum')
).reset_index()
agg_df_3['has_intervention'] = (agg_df_3['intervened_events'] > 0).astype(int)

years_3 = sorted(agg_df_3['year'].unique())
y0_3 = years_3[0]

fig3 = go.Figure()
fig3.add_trace(go.Scattermapbox(lat=[0], lon=[0], mode='markers', name='All Protests (Baseline)', marker=dict(size=8, color='#3498db'), visible=False, showlegend=True, hoverinfo='skip'))
fig3.add_trace(go.Scattermapbox(lat=[0], lon=[0], mode='markers', name='State Interventions', marker=dict(size=8, color='#E74C3C'), visible=False, showlegend=True, hoverinfo='skip'))

for c in target_countries:
    df_y0_c = agg_df_3[(agg_df_3['year'] == y0_3) & (agg_df_3['country'] == c)]
    df_y0_red = df_y0_c[df_y0_c['has_intervention'] == 1]
    fig3.add_trace(go.Scattermapbox(
        lat=df_y0_c['latitude'], lon=df_y0_c['longitude'], mode='markers',
        marker=dict(size=7, color=country_colors_3[c], opacity=0.45), showlegend=False,
        customdata=df_y0_c[['location', 'country', 'total_events']],
        hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br>Total Events: %{customdata[2]}<extra></extra>", visible=False
    ))
    fig3.add_trace(go.Scattermapbox(
        lat=df_y0_red['latitude'], lon=df_y0_red['longitude'], mode='markers',
        marker=dict(size=11, color='#E74C3C', opacity=0.92), showlegend=False,
        customdata=df_y0_red[['location', 'country', 'intervened_events']],
        hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br><b>Intervened Events: %{customdata[2]}</b><extra></extra>", visible=False
    ))

frames_3 = []
for y in years_3:
    frame_data = [go.Scattermapbox(lat=[0], lon=[0], visible=False), go.Scattermapbox(lat=[0], lon=[0], visible=False)]
    for c in target_countries:
        df_y_c = agg_df_3[(agg_df_3['year'] == y) & (agg_df_3['country'] == c)]
        df_y_red = df_y_c[df_y_c['has_intervention'] == 1]
        frame_data.append(go.Scattermapbox(lat=df_y_c['latitude'], lon=df_y_c['longitude'], mode='markers', marker=dict(size=7, color=country_colors_3[c], opacity=0.45), customdata=df_y_c[['location', 'country', 'total_events']], hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br>Total Events: %{customdata[2]}<extra></extra>"))
        frame_data.append(go.Scattermapbox(lat=df_y_red['latitude'], lon=df_y_red['longitude'], mode='markers', marker=dict(size=11, color='#E74C3C', opacity=0.92), customdata=df_y_red[['location', 'country', 'intervened_events']], hovertemplate="<b>%{customdata[0]}</b><br>Country: %{customdata[1]}<br><b>Intervened Events: %{customdata[2]}</b><extra></extra>"))
    frames_3.append(go.Frame(data=frame_data, name=str(y)))

fig3.frames = frames_3

def get_vis_3(country_list):
    vis = [False, False]
    for c in target_countries:
        vis.extend([True, True] if c in country_list else [False, False])
    return vis

combo_buttons = [
    dict(label="🌐 All Countries", method="restyle", args=[{"visible": get_vis_3(target_countries)}]),
    dict(label="⚔️ West (UK, FR, DE)", method="restyle", args=[{"visible": get_vis_3(['United Kingdom', 'France', 'Germany'])}]),
    dict(label="⚔️ East (CZ, SK)", method="restyle", args=[{"visible": get_vis_3(['Czech Republic', 'Slovakia'])}]),
]
for c in target_countries:
    combo_buttons.append(dict(label=f"🔹 Only {c}", method="restyle", args=[{"visible": get_vis_3([c])}]))

fig3.update_layout(
    autosize=True,
    uirevision='locked',
    legend=dict(orientation="h", y=0.97, x=0.5, xanchor="center", bgcolor='rgba(255,255,255,0.9)', bordercolor='lightgray', borderwidth=1, font=dict(size=12)),
    updatemenus=[
        dict(buttons=combo_buttons, direction="down", x=0.0, y=1, xanchor="left", yanchor="top", showactive=True, bgcolor="white", bordercolor="#ccc"),
        dict(type="buttons", direction="left", x=0, y=-0.07, xanchor="left", yanchor="top",
             buttons=[
                 dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 1500, "redraw": True}, "fromcurrent": True, "transition": {"duration": 500}}]),
                 dict(label="〓 Pause", method="animate", args=[[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}}])
             ], bgcolor="white", bordercolor="#ccc")
    ],
    sliders=[{"active": 0, "y": 0.05, "x": 0.13, "len": 0.85, "currentvalue": {"prefix": "Year: ", "font": {"size": 16, "color": "#0d6efd"}},
             "steps": [{"args": [[str(y)], {"frame": {"duration": 800, "redraw": True}, "mode": "immediate", "transition": {"duration": 300}}], "label": str(y), "method": "animate"} for y in years_3]}],
    mapbox=dict(style="carto-positron", zoom=4, center=dict(lat=50, lon=10)),
    margin={"r": 0, "t": 100, "l": 0, "b": 0},
    height=700,
    title=dict(text="<b>Comparison of Cross-border Space Deterrence (Based on Annual Aggregation)</b><b><br><sup>Different colors represent different countries, with red dots indicating state intervention events</sup>", font=dict(size=20), y=0.91, x=0.5, xanchor='center')
)


# =============================================================================
# Plot 4: Capital vs Other Regions — Dumbbell Chart (Violence Gap)
# =============================================================================
print("Generating Plot 4...")

df_micro_4 = pd.read_csv('Micro_With_VDem.csv')
df_micro_4 = df_micro_4[df_micro_4['country'].isin(target_countries)].copy()
df_micro_4['year'] = pd.to_datetime(df_micro_4['event_date']).dt.year
df_micro_4['location'] = df_micro_4['location'].astype(str)
df_micro_4['capital_name'] = df_micro_4['country'].map(capitals)
df_micro_4['is_capital'] = np.where(
    df_micro_4.apply(lambda row: row['capital_name'].lower() in row['location'].lower() if pd.notna(row['capital_name']) and pd.notna(row['location']) else False, axis=1), 1, 0
)
df_micro_4['is_intervened'] = df_micro_4['sub_event_type'].isin(['Protest with intervention', 'Excessive force against protesters']).astype(int)
df_micro_4['Region'] = df_micro_4['is_capital'].map({1: 'Capital City', 0: 'Other Regions'})

df_spatial = df_micro_4.groupby(['country', 'Region']).agg(total=('event_id_cnty', 'count'), intervened=('is_intervened', 'sum')).reset_index()
df_spatial['rate'] = df_spatial['intervened'] / df_spatial['total']
df_wide = df_spatial.pivot(index='country', columns='Region', values='rate').reset_index()
for col in ['Capital City', 'Other Regions']:
    if col not in df_wide.columns:
        df_wide[col] = 0
df_wide = df_wide.fillna(0)
df_wide['gap'] = df_wide['Capital City'] - df_wide['Other Regions']
df_wide['gap_label'] = df_wide['gap'].map(lambda x: f"{x:+.1%}")
df_wide = df_wide.sort_values('gap', ascending=True).reset_index(drop=True)

fig4 = go.Figure()

for _, row in df_wide.iterrows():
    fig4.add_trace(go.Scatter(
        x=[row['Other Regions'], row['Capital City']], y=[row['country'], row['country']],
        mode='lines', line=dict(color='rgba(120,120,120,0.55)', width=3), hoverinfo='skip', showlegend=False
    ))

fig4.add_trace(go.Scatter(
    x=df_wide['Other Regions'], y=df_wide['country'], mode='markers+text', name='Other Regions',
    marker=dict(size=14, color='#AED6F1', line=dict(color='white', width=1.5)),
    text=[f"{v:.1%}" for v in df_wide['Other Regions']], textposition='middle left',
    customdata=df_wide[['gap_label']],
    hovertemplate="<b>%{y}</b><br>Other Regions: %{x:.2%}<br>Gap: %{customdata[0]}<extra></extra>"
))

fig4.add_trace(go.Scatter(
    x=df_wide['Capital City'], y=df_wide['country'], mode='markers+text', name='Capital City',
    marker=dict(size=16, color='#E74C3C', line=dict(color='white', width=1.5)),
    text=[f"{v:.1%}" for v in df_wide['Capital City']], textposition='middle right',
    customdata=df_wide[['gap_label']],
    hovertemplate="<b>%{y}</b><br>Capital City: %{x:.2%}<br>Gap: %{customdata[0]}<extra></extra>"
))

fig4.add_trace(go.Scatter(
    x=(df_wide['Other Regions'] + df_wide['Capital City']) / 2, y=df_wide['country'],
    mode='text', text=df_wide['gap_label'], textposition='top center',
    textfont=dict(size=11, color='dimgray'), showlegend=False, hoverinfo='skip'
))

fig4.update_layout(
    title="<b>Capital —— Local: The Violence Gap</b><br>"
    "<sup>The longer the line segment, the more concentrated the<br>"
    "state's coercive intervention is at the political center</sup>",
    xaxis_title="Mandatory intervention rate", yaxis_title="Countries",
    plot_bgcolor="white", paper_bgcolor="white", height=560,
    margin=dict(l=120, r=30, t=130, b=70),
    legend=dict(orientation='h', y=1.08, x=0.5, xanchor='center')
)
fig4.update_xaxes(tickformat=".0%", showgrid=True, gridcolor='rgba(0,0,0,0.08)', zeroline=False)
fig4.update_yaxes(showgrid=False, categoryorder='array', categoryarray=df_wide['country'])


# =============================================================================
# Plot 5: Annual Gap Trend Chart (Capital vs Other Regions)
# =============================================================================
print("Generating Plot 5...")

df_year_5 = df_micro_4.groupby(['country', 'year', 'Region']).agg(total=('event_id_cnty', 'count'), intervened=('is_intervened', 'sum')).reset_index()
df_year_5['rate'] = df_year_5['intervened'] / df_year_5['total']
df_year_wide = df_year_5.pivot(index=['country', 'year'], columns='Region', values='rate').reset_index()
for col in ['Capital City', 'Other Regions']:
    if col not in df_year_wide.columns:
        df_year_wide[col] = 0
df_year_wide = df_year_wide.fillna(0)
df_year_wide['Gap'] = df_year_wide['Capital City'] - df_year_wide['Other Regions']
years_5 = sorted(df_year_wide['year'].unique())

def subset_country_5(country_name):
    if country_name == 'All Countries':
        temp = df_year_wide.groupby('year')[['Capital City', 'Other Regions', 'Gap']].mean().reset_index()
    else:
        temp = df_year_wide[df_year_wide['country'] == country_name].copy()
    return temp.sort_values('year')

df_init_5 = subset_country_5('All Countries')

fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=df_init_5['year'], y=df_init_5['Capital City'], mode='lines+markers', name='Capital City', line=dict(color='#E74C3C', width=3), marker=dict(size=8), hovertemplate="Year: %{x}<br>Capital City: %{y:.2%}<extra></extra>"))
fig5.add_trace(go.Scatter(x=df_init_5['year'], y=df_init_5['Other Regions'], mode='lines+markers', name='Other Regions', line=dict(color='#3498DB', width=3), marker=dict(size=8), hovertemplate="Year: %{x}<br>Other Regions: %{y:.2%}<extra></extra>"))
fig5.add_trace(go.Scatter(x=df_init_5['year'], y=df_init_5['Gap'], mode='lines+markers', name='Gap (Capital - Other)', line=dict(color='#2C3E50', width=3, dash='dash'), marker=dict(size=7), hovertemplate="Year: %{x}<br>Gap: %{y:.2%}<extra></extra>"))

buttons_5 = []
for c in ['All Countries'] + target_countries:
    df_c = subset_country_5(c)
    buttons_5.append(dict(label=c, method='update', args=[{'x': [df_c['year'], df_c['year'], df_c['year']], 'y': [df_c['Capital City'], df_c['Other Regions'], df_c['Gap']]}, {'title': f"<b>Capital — Local, Annual changes in intervention gap</b><br><sup>Current view: {c}</sup>"}]))

fig5.update_layout(
    title="<b>Capital — Local, Annual changes in intervention gap</b><br><sup>Current view: All Countries</sup>",
    xaxis_title="Year", yaxis_title="Intervention rate / Difference",
    plot_bgcolor="white", paper_bgcolor="white", height=560,
    margin=dict(l=80, r=60, t=110, b=70),
    legend=dict(orientation='h', y=1.08, x=0.65, xanchor='center'),
    updatemenus=[dict(buttons=buttons_5, direction='down', x=-0.07, y=1.1, xanchor='left', yanchor='top', showactive=True, bgcolor='white', bordercolor='#ccc')],
    shapes=[dict(type='line', x0=min(years_5), x1=max(years_5), y0=0, y1=0, line=dict(color='gray', width=1.2, dash='dot'))]
)
fig5.update_xaxes(tickmode='linear', dtick=1, showgrid=False)
fig5.update_yaxes(tickformat=".0%", showgrid=True, gridcolor='rgba(0,0,0,0.08)', zeroline=False)


# =============================================================================
# Plot 6: Dynamic Pressure Test Scatter Plot
# =============================================================================
print("Generating Plot 6...")

df_panel_6 = pd.read_csv('Final_Merged_Panel.csv')
df_panel_6['year_month'] = pd.to_datetime(df_panel_6['year_month'])
df_panel_6['Year'] = df_panel_6['year_month'].dt.year
df_panel_6['Quarter'] = df_panel_6['year_month'].dt.quarter
df_panel_6['Time_Quarter'] = df_panel_6['Year'].astype(str) + '-Q' + df_panel_6['Quarter'].astype(str)

baseline_years_6 = [2021, 2023]
df_baseline_6 = df_panel_6[df_panel_6['Year'].isin(baseline_years_6)].groupby('country')['crackdown_rate'].mean().reset_index()
df_baseline_6.columns = ['country', 'baseline_rate']

df_quarterly_6 = df_panel_6.groupby(['country', 'Time_Quarter']).agg(
    current_rate=('crackdown_rate', 'mean'),
    total_events=('total_protests', 'sum'),
    v2x_libdem=('v2x_libdem', 'mean'),
    v2x_rule=('v2x_rule', 'mean'),
    v2x_polyarchy=('v2x_polyarchy', 'mean'),
    v2x_freexp_altinf=('v2x_freexp_altinf', 'mean')
).reset_index()

df_dynamic_6 = pd.merge(df_quarterly_6, df_baseline_6, on='country').dropna()

MIN_SIZE, MAX_SIZE = 20, 75
global_min_e = df_dynamic_6['total_events'].min()
global_max_e = df_dynamic_6['total_events'].max()

def calc_smooth_size(val):
    if global_max_e == global_min_e:
        return (MIN_SIZE + MAX_SIZE) / 2
    return MIN_SIZE + ((val - global_min_e) / (global_max_e - global_min_e)) * (MAX_SIZE - MIN_SIZE)

df_dynamic_6['smooth_size'] = df_dynamic_6['total_events'].apply(calc_smooth_size)

quarters_6 = sorted(df_dynamic_6['Time_Quarter'].unique())
q0_6 = quarters_6[0]

hover_cols_6 = ['baseline_rate', 'current_rate', 'total_events', 'v2x_libdem', 'v2x_rule', 'v2x_polyarchy', 'v2x_freexp_altinf']

hover_template_6 = """
<span style="font-size:11px;">--- <b>Crisis Data</b> ---</span><br>
Baseline: %{customdata[0]:.4f} | Current Q: %{customdata[1]:.4f}<br>
Protest Volume: %{customdata[2]:.0f}<br>
<span style="font-size:11px;">--- <b>V-Dem Context</b> ---</span><br>
Liberal Dem: %{customdata[3]:.3f} | Rule of Law: %{customdata[4]:.3f}<br>
Electoral: %{customdata[5]:.3f} | Free Exp: %{customdata[6]:.3f}
<extra></extra>
"""

marker_base_6 = dict(line=dict(width=2.5, color='#2C3E50'), opacity=0.80)

fig6 = go.Figure()

for i, c in enumerate(df_dynamic_6['country'].unique()):
    df_q0_c = df_dynamic_6[(df_dynamic_6['Time_Quarter'] == q0_6) & (df_dynamic_6['country'] == c)]
    m_style = marker_base_6.copy()
    m_style['size'] = df_q0_c['smooth_size'].values[0]
    fig6.add_trace(go.Scatter(
        x=[df_q0_c['baseline_rate'].values[0]], y=[df_q0_c['current_rate'].values[0]],
        mode='markers+text', text=c, textposition='top center', textfont=dict(size=12),
        marker=m_style, name=c,
        customdata=df_q0_c[hover_cols_6],
        hovertemplate=hover_template_6
    ))

frames_6 = []
for q in quarters_6:
    frame_data = []
    for c in df_dynamic_6['country'].unique():
        df_q_c = df_dynamic_6[(df_dynamic_6['Time_Quarter'] == q) & (df_dynamic_6['country'] == c)]
        m_style_frame = marker_base_6.copy()
        if not df_q_c.empty:
            m_style_frame['size'] = df_q_c['smooth_size'].values[0]
            frame_data.append(go.Scatter(
                x=[df_q_c['baseline_rate'].values[0]], y=[df_q_c['current_rate'].values[0]],
                marker=m_style_frame, customdata=df_q_c[hover_cols_6], hovertemplate=hover_template_6
            ))
        else:
            frame_data.append(go.Scatter(x=[None], y=[None]))
    frames_6.append(go.Frame(data=frame_data, name=q))

fig6.frames = frames_6

fig6.add_shape(type="line", x0=-0.25, y0=-0.3, x1=0.25, y1=0.3, line=dict(color="gray", width=3, dash="dash"))

fig6.update_layout(
    uirevision='locked',
    updatemenus=[
        dict(type="buttons", direction="left", x=-0.025, y=-0.08, xanchor="left", yanchor="top",
             buttons=[
                 dict(label="▶ Play", method="animate", args=[None, {"frame": {"duration": 1200, "redraw": True}, "fromcurrent": True, "transition": {"duration": 400, "easing": "cubic-in-out"}}]),
                 dict(label="〓 Pause", method="animate", args=[[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}}])
             ], bgcolor="white", bordercolor="#ccc", font=dict(size=14))
    ],
    sliders=[{
        "active": 0, "y": 0.0, "x": 0.15, "len": 0.85,
        "currentvalue": {"prefix": "Quarter: ", "font": {"size": 16, "color": "#0d6efd"}},
        "steps": [{"args": [[q], {"frame": {"duration": 600, "redraw": True}, "mode": "immediate", "transition": {"duration": 300}}], "label": q, "method": "animate"} for q in quarters_6]
    }],
    plot_bgcolor="white",
    xaxis=dict(title="<b>Normal Baseline Intervention Rate</b>", linecolor='#AAAAAA', linewidth=1, mirror=True, gridcolor='#E5E7EB', range=[-0.05, 0.25], zeroline=True, zerolinecolor='black', zerolinewidth=2),
    yaxis=dict(title="<b>Current Quarter Intervention Rate</b>", linecolor='#AAAAAA', linewidth=1, mirror=True, gridcolor='#E5E7EB', range=[-0.05, 0.3], zeroline=True, zerolinecolor='black', zerolinewidth=2),
    legend=dict(orientation="h", y=1.08, x=0.0, xanchor="left", bgcolor='rgba(255,255,255,0.9)', bordercolor='lightgray', borderwidth=1),
    annotations=[dict(x=0.2, y=0.160, text="<b>45° Baseline </b><br>Points on the line = Crisis had no effect<br><span style='color:#c0392b'><b>Points above</b> = Disproportionate Violence Amplification</span>", showarrow=False, font=dict(color="#555555", size=11), align='left')],
    title=dict(text="<b>Dynamic Pressure Test with Multi-Dimensional Context</b><br>Hover to inspect V-Dem indices alongside crisis deviations", x=0.5, y=0.98, xanchor='center'),
    height=700, margin=dict(l=70, r=40, t=100, b=60)
)


# =============================================================================
# Plot 7: Spatial-Temporal Stress Matrix Heatmap
# =============================================================================
print("Generating Plot 7...")

df_panel_7 = pd.read_csv('Final_Merged_Panel.csv')
df_panel_7['year_month'] = pd.to_datetime(df_panel_7['year_month'])
df_panel_7['year'] = df_panel_7['year_month'].dt.year
df_panel_7['month'] = df_panel_7['year_month'].dt.month

months_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
countries_7 = sorted(df_panel_7['country'].unique())
years_7 = sorted(df_panel_7['year'].unique())
y0_7 = years_7[0]

baseline_years_7 = [y for y in years_7 if y not in [2020, 2022]]
df_baseline_7 = df_panel_7[df_panel_7['year'].isin(baseline_years_7)].groupby('country', as_index=False).agg(baseline_rate=('crackdown_rate', 'mean'))
df_panel_7 = df_panel_7.merge(df_baseline_7, on='country', how='left')
df_panel_7['deviation_from_baseline'] = df_panel_7['crackdown_rate'] - df_panel_7['baseline_rate']

full_index_7 = pd.MultiIndex.from_product([countries_7, years_7, range(1, 13)], names=['country', 'year', 'month'])
df_monthly_7 = df_panel_7.groupby(['country', 'year', 'month'], as_index=False).agg(crackdown_rate=('crackdown_rate', 'mean'), baseline_rate=('baseline_rate', 'mean'))
df_monthly_7 = df_monthly_7.set_index(['country', 'year', 'month']).reindex(full_index_7).reset_index()
df_monthly_7['baseline_rate'] = df_monthly_7.groupby('country')['baseline_rate'].transform(lambda x: x.ffill().bfill())
df_monthly_7['crackdown_rate'] = df_monthly_7['crackdown_rate'].fillna(0)
df_monthly_7['deviation_from_baseline'] = df_monthly_7['crackdown_rate'] - df_monthly_7['baseline_rate']

yearly_matrices_7 = {}
yearly_hover_7 = {}
for y in years_7:
    df_y = df_monthly_7[df_monthly_7['year'] == y]
    matrix = df_y.pivot(index='country', columns='month', values='deviation_from_baseline').reindex(index=countries_7, columns=range(1, 13))
    raw_matrix = df_y.pivot(index='country', columns='month', values='crackdown_rate').reindex(index=countries_7, columns=range(1, 13))
    baseline_matrix = df_y.pivot(index='country', columns='month', values='baseline_rate').reindex(index=countries_7, columns=range(1, 13))
    yearly_matrices_7[y] = matrix.values
    yearly_hover_7[y] = np.dstack([raw_matrix.values, baseline_matrix.values, matrix.values])

all_dev_7 = np.concatenate([yearly_matrices_7[y].flatten() for y in years_7])
all_dev_7 = all_dev_7[~np.isnan(all_dev_7)]
abs_limit_7 = np.nanpercentile(np.abs(all_dev_7), 95)
zmin_7, zmax_7 = -abs_limit_7, abs_limit_7

fig7 = go.Figure()
fig7.add_trace(go.Heatmap(
    z=yearly_matrices_7[y0_7], x=months_labels, y=countries_7,
    customdata=yearly_hover_7[y0_7],
    colorscale='RdBu_r', zmid=0, zmin=zmin_7, zmax=zmax_7,
    colorbar=dict(title="Deviation from Baseline", tickformat=".3f"),
    hovertemplate="<b>%{y}</b><br>Month: %{x}<br>Current rate: %{customdata[0]:.4f}<br>Baseline rate: %{customdata[1]:.4f}<br><b>Deviation: %{customdata[2]:+.4f}</b><extra></extra>"
))

crisis_notes = {
    2020: "COVID-19 shock year: watch for sudden positive deviations in spring.",
    2022: "Russia-Ukraine War shock year: watch for renewed deviations from normal restraint.",
    2023: "Energy crisis year: Lack of red indicates 'Baseline Reset' (prior shocks normalized high violence)."
}

frames_7 = []
for y in years_7:
    note_text = crisis_notes.get(y, "Normal observation year: deviations should fluctuate closely around zero.")
    font_color = '#c0392b' if y in [2020, 2022] else '#7f8c8d'
    frames_7.append(go.Frame(
        data=[go.Heatmap(z=yearly_matrices_7[y], customdata=yearly_hover_7[y])],
        layout=dict(annotations=[dict(x=1.0, y=1.14, xref='paper', yref='paper', text=f"<b>Insight ({y}):</b> {note_text}", showarrow=False, xanchor='right', font=dict(size=12, color=font_color))]),
        name=str(y)
    ))
fig7.frames = frames_7

fig7.update_layout(
    updatemenus=[
        dict(type="buttons", direction="left", x=-0.11, y=-0.1, xanchor="left", yanchor="top",
             buttons=[
                 dict(label="▶ Play Years", method="animate", args=[None, {"frame": {"duration": 1500, "redraw": True}, "fromcurrent": True, "transition": {"duration": 400}}]),
                 dict(label="〓 Pause", method="animate", args=[[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}}])
             ], bgcolor="white", bordercolor="#ccc")
    ],
    sliders=[{
        "active": 0, "y": 0.0, "x": 0.15, "len": 0.85,
        "currentvalue": {"prefix": "Year: ", "font": {"size": 16, "color": "#0d6efd"}},
        "steps": [{"args": [[str(y)], {"frame": {"duration": 800, "redraw": True}, "mode": "immediate"}], "label": str(y), "method": "animate"} for y in years_7]
    }],
    plot_bgcolor="white",
    xaxis=dict(title="<b>Month</b>", linecolor='#2C3E50', linewidth=1, side='top'),
    yaxis=dict(title="<b>Country</b>", linecolor='#2C3E50', linewidth=1, tickfont=dict(size=13), autorange='reversed'),
    title=dict(text="<b>Dynamic Stress-Test Heatmap: Deviations from Normal Patterns</b><br><sup>Red = above baseline; Blue = below baseline. Watch how the top-right insight text changes during crisis years.</sup>", x=0.5, y=0.97, xanchor='center'),
    height=700, margin=dict(l=110, r=40, t=110, b=50)
)


# =============================================================================
# INTEGRATED DASHBOARD HTML ASSEMBLY
# =============================================================================
print("\nAssembling integrated dashboard HTML...")

# Convert each figure to inline HTML (no full HTML wrapper, no plotly.js)
plot1_html = pio.to_html(fig1, include_plotlyjs=False, full_html=False)
plot2_html = pio.to_html(fig2, include_plotlyjs=False, full_html=False)
plot3_html = pio.to_html(fig3, include_plotlyjs=False, full_html=False)
plot4_html = pio.to_html(fig4, include_plotlyjs=False, full_html=False)
plot5_html = pio.to_html(fig5, include_plotlyjs=False, full_html=False)
plot6_html = pio.to_html(fig6, include_plotlyjs=False, full_html=False)
plot7_html = pio.to_html(fig7, include_plotlyjs=False, full_html=False)

dashboard_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>European Narrative Line: Protests, Democracy and State Violence</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0-alpha1/dist/css/bootstrap.min.css" rel="stylesheet">
    <script type="text/javascript">window.PlotlyConfig = {{MathJaxConfig: 'local'}};</script>
    <script charset="utf-8" src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
    <style>
        body {{
            background-color: #f4f6f9;
            padding: 25px;
            font-family: 'Segoe UI', Tahoma, sans-serif;
            color: #1f2937;
        }}
        .main-card {{
            border-radius: 10px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.06);
            margin-bottom: 28px;
            border: 1px solid #e0e4e8;
            background-color: #ffffff;
        }}
        .main-card-body {{
            padding: 28px;
        }}
        h1 {{
            color: #1a202c;
            font-weight: 700;
            font-size: 2.1rem;
            margin-bottom: 15px;
        }}
        .section-title {{
            color: #2d3748;
            font-weight: 700;
            font-size: 1.45rem;
            margin-bottom: 20px;
            border-bottom: 3px solid #3182ce;
            padding-bottom: 10px;
        }}
        .sub-section-title {{
            color: #2d3748;
            font-weight: 600;
            font-size: 1.2rem;
            margin-bottom: 16px;
            border-bottom: 2px solid #edf2f7;
            padding-bottom: 8px;
        }}
        .header-info {{
            color: #4a5568;
            font-size: 1rem;
            margin-bottom: 0;
            line-height: 1.7;
        }}
        .stat-card {{
            padding: 18px 12px;
            border: 1px solid #e2e8f0;
            border-radius: 10px;
            background: #f8fafc;
            text-align: center;
        }}
        .stat-card h2 {{
            font-weight: 800;
            margin-bottom: 5px;
        }}
        .insight-box {{
            background-color: #f8fafc;
            border: 1px solid #e2e8f0;
            border-left: 4px solid #3182ce;
            border-radius: 6px;
            padding: 16px 18px;
            height: 100%;
            font-size: 0.95rem;
            line-height: 1.7;
            color: #4a5568;
        }}
        .method-title {{
            border-bottom: 3px solid #3182ce;
            padding-bottom: 12px;
            margin-bottom: 22px;
            color: #2d3748;
            font-weight: 700;
            font-size: 1.35rem;
        }}
        .method-subtitle {{
            color: #2b6cb0;
            font-size: 1.08rem;
            font-weight: 600;
            margin-bottom: 10px;
        }}
        .method-text {{
            color: #4a5568;
            font-size: 0.95rem;
            line-height: 1.75;
        }}
        .rq-badge {{
            display: inline-block;
            padding: 4px 14px;
            border-radius: 20px;
            font-size: 0.85rem;
            font-weight: 600;
            margin-right: 8px;
            margin-bottom: 6px;
        }}
        .rq1-badge {{ background-color: #ebf5fb; color: #2980b9; border: 1px solid #aed6f1; }}
        .rq2-badge {{ background-color: #fef9e7; color: #d4ac0d; border: 1px solid #f9e79f; }}
        .rq3-badge {{ background-color: #fdedec; color: #c0392b; border: 1px solid #f5b7b1; }}
        .plot-container {{
            border-radius: 10px;
            background: white;
            border: 1px solid #e5e7eb;
            margin-bottom: 0;
        }}
        .summary-box {{
            background: linear-gradient(135deg, #f8fafc 0%, #ebf5fb 100%);
            border: 1px solid #d4e6f1;
            border-radius: 8px;
            padding: 20px;
            height: 100%;
        }}
        .summary-box h5 {{
            color: #2c3e50;
            font-weight: 700;
            font-size: 1.1rem;
            margin-bottom: 10px;
        }}
        .summary-box p {{
            color: #4a5568;
            font-size: 0.93rem;
            line-height: 1.7;
            margin-bottom: 0;
        }}
        .divider {{
            border: 0;
            height: 2px;
            background: linear-gradient(to right, #3182ce, #63b3ed, #3182ce);
            margin: 10px 0 28px 0;
        }}
        .footer-text {{
            text-align: center;
            color: #718096;
            font-size: 0.85rem;
            padding: 20px 0 10px 0;
        }}
    </style>
</head>
<body>

<div class="container-fluid">

    <!-- ================================================================== -->
    <!-- SECTION 1: Header — Course Info, Personal Info, Dataset Info        -->
    <!-- ================================================================== -->
    <div class="main-card">
        <div class="main-card-body">
            <h1>European Narrative Line: Protests, Democracy and State Violence</h1>
            <p class="header-info">
                <strong>Student Name:</strong> Jiang Shuo &nbsp;|&nbsp;
                <strong>UID:</strong> 3036336429 &nbsp;|&nbsp;
                <strong>Course:</strong> POLI 3148 Data Science in Politics and Public Administration
                <br>
                <strong>Dataset:</strong> <a href="https://acleddata.com/" target="_blank">ACLED</a> (Micro-event protest data, 2018–2024) &amp; <a href="https://www.v-dem.net/" target="_blank">V-Dem v16</a> (Macro-institutional indices)
            </p>

            <div class="row mt-4">
                <div class="col-md-3">
                    <div class="stat-card">
                        <h2 style="color:#2980b9;">5</h2>
                        <span class="text-muted">European Countries</span>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card">
                        <h2 style="color:#27ae60;">2+3</h2>
                        <span class="text-muted">Number of Datasets (Including processed)</span>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card">
                        <h2 style="color:#e67e22;">7</h2>
                        <span class="text-muted">Analytical Visualizations</span>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card">
                        <h2 style="color:#c0392b;">5</h2>
                        <span class="text-muted">Actual Usable Data (Year)</span>
                    </div>
                </div>
            </div>
        </div>
    </div>

    <!-- ================================================================== -->
    <!-- SECTION 2: Research Design & Methodology + 3 RQs                    -->
    <!-- ================================================================== -->
    <div class="main-card" style="border-left: 4px solid #3182ce;">
        <div class="main-card-body">
            <h3 class="method-title">Research Design &amp; Methodology</h3>
            <div class="row">
                <div class="col-md-6 mb-3 mb-md-0">
                    <h5 class="method-subtitle">Data Fusion Strategy &amp; Dependent Variable</h5>
                    <p class="method-text">
                        This project bridges <strong>micro-level event data</strong> (ACLED) with <strong>macro-level institutional indices</strong> (V-Dem). By calculating a <strong>State Intervention Rate</strong> (<em>protests interrupted by police / total protests</em>) on a monthly panel for five European countries (2018–2024), we bypass the limitations of absolute fatality counts and isolate the state's <em>propensity for force</em> as a measurable dependent variable. This design addresses the endogeneity of state repression. By utilizing a panel data approach with fixed effects, that can control for country-specific constants (like political culture) and focus purely on how fluctuations in institutional quality (V-Dem) drive changes in the state's tactical response to its citizens.
                    </p>
                    <p class="method-text mt-2">
                        <strong>Independent Variables (V-Dem):</strong><br>
                        &bull; <code>v2x_libdem</code> — Liberal Democracy Index (institutional constraint)<br>
                        &bull; <code>v2x_rule</code> — Rule of Law Index (police independence)<br>
                        &bull; <code>v2x_freexp_altinf</code> — Expression &amp; Association Freedom (protest space)
                    </p>
                </div>
                <div class="col-md-6">
                    <h5 class="method-subtitle">Research Questions (RQs)</h5>
                    <p class="method-text" style="font-size: 0.93rem;">
                        <span class="rq-badge rq1-badge">RQ1</span>
                        <strong>Threshold Effect:</strong> Does democratic backsliding lower the state's tolerance threshold for dissent in a <em>non-linear</em> fashion? Is there a critical breaking point beyond which institutional restraints collapse?<br><br>

                        <span class="rq-badge rq2-badge">RQ2</span>
                        <strong>Spatial Deterrence:</strong> How does institutional decay alter the <em>spatial distribution</em> of state violence? Do backsliding regimes concentrate coercive force in political centers to maximize deterrence?<br><br>

                        <span class="rq-badge rq3-badge">RQ3</span>
                        <strong>Crisis Amplifier:</strong> Do exogenous crises (Covid-19, Russia-Ukraine War) act as <em>pressure tests</em> that expose and accelerate underlying institutional fragility, producing irreversible shifts in state violence baselines?
                    </p>
                </div>
            </div>
        </div>
    </div>

    <hr class="divider">

    <!-- ================================================================== -->
    <!-- SECTION 3: RQ1 — Threshold Effect (Plot 1 & Plot 2)                -->
    <!-- ================================================================== -->
    <div class="main-card" style="border-left: 4px solid #2980b9;">
        <div class="main-card-body">
            <div class="d-flex align-items-center mb-2">
                <span class="rq-badge rq1-badge" style="font-size:1rem; padding:6px 18px;">RQ1</span>
                <h3 class="section-title mb-0" style="border-bottom:none; margin-bottom:0;">Threshold Effect: Democratic Erosion &amp; Non-Linear Violence</h3>
            </div>
            <p class="method-text mb-0">
                Does democratic decline increase state violence linearly, or does it destroy institutional buffers until a critical threshold is crossed? These two plots probe the non-linear relationship between V-Dem scores and state intervention rates, revealing the "breaking point" in democratic restraint.
            </p>
        </div>
    </div>

    <div class="main-card">
        <div class="main-card-body">
            <h4 class="sub-section-title">Plot 1: Spatial Distribution of Protests &amp; State Intervention Hotspots</h4>
            <div class="plot-container" style="min-height: 700px;">
                {plot1_html}
            </div>
            <div class="insight-box mt-3">
                <strong>Response to RQ1:</strong> This animated map reveals the <em>spatial-temporal unevenness</em> of state intervention across five European countries. By toggling between countries and playing the quarterly timeline, observe that high-intervention clusters (red markers) concentrate disproportionately in backsliding democracies (Czech Republic, Slovakia) during crisis quarters — confirming that democratic erosion does not produce uniform violence but creates <strong>localized hotspots</strong> when institutional thresholds are breached. The V-Dem scores shown on hover contextualize each location's democratic quality.
            </div>
        </div>
    </div>

    <div class="main-card">
        <div class="main-card-body">
            <h4 class="sub-section-title">Plot 2: Cross-National Repression Rate Trajectories</h4>
            <div class="plot-container">
                {plot2_html}
            </div>
            <div class="insight-box mt-3">
                <strong>Response to RQ1:</strong> Longitudinal tracking highlights <strong>heterogeneous institutional elasticity</strong>. While high-resilience countries (Germany/UK) maintain a flat baseline of restraint regardless of external events, backsliding countries (Czech Republic) exhibit sudden, high-amplitude pulses — visually confirming the volatility predicted by the threshold model. The dropdown menu allows switching to individual country views where area fills and peak markers dramatically illustrate the "cardiogram-like" instability of fragile democracies versus the "flatline" stability of resilient ones.
            </div>
        </div>
    </div>

    <hr class="divider">

    <!-- ================================================================== -->
    <!-- SECTION 4: RQ2 — Spatial Deterrence (Plot 3, Plot 4, Plot 5)       -->
    <!-- ================================================================== -->
    <div class="main-card" style="border-left: 4px solid #d4ac0d;">
        <div class="main-card-body">
            <div class="d-flex align-items-center mb-2">
                <span class="rq-badge rq2-badge" style="font-size:1rem; padding:6px 18px;">RQ2</span>
                <h3 class="section-title mb-0" style="border-bottom:none; margin-bottom:0;">Spatial Deterrence: The Geography of State Violence</h3>
            </div>
            <p class="method-text mb-0">
                When institutions decay, does the state abandon broad territorial policing in favor of concentrated, highly visible deterrence in political centers? These three plots quantify the "Violence Gap" between capital cities and peripheral regions, testing whether backsliding regimes redistribute force spatially.
            </p>
        </div>
    </div>

    <!-- Plot 3: Full Width -->
    <div class="main-card">
        <div class="main-card-body">
            <h4 class="sub-section-title">Plot 3: Cross-border Spatial Deterrence Comparison (Annual Aggregation)</h4>
            <div class="plot-container">
                {plot3_html}
            </div>
            <div class="insight-box mt-3">
                <strong>Response to RQ2:</strong> This animated spatial map directly visualizes <strong>where state interventions occur</strong>. Different colors represent different countries; red dots mark state intervention events. By playing the yearly timeline and toggling between "East (CZ, SK)" and "West (UK, FR, DE)," observe that in backsliding regimes, red intervention markers concentrate heavily in capital regions (Prague, Bratislava), while democratic countries show a more dispersed pattern — providing direct spatial evidence for the <em>centralized deterrence hypothesis</em>.
            </div>
        </div>
    </div>

    <!-- Plot 4 & 5: Side by Side -->
    <div class="row">
        <div class="col-lg-6">
            <div class="main-card">
                <div class="main-card-body">
                    <h4 class="sub-section-title">Plot 4: The Violence Gap — Capital vs. Other Regions</h4>
                    <div class="plot-container">
                        {plot4_html}
                    </div>
                    <div class="insight-box mt-3">
                        <strong>Response to RQ2:</strong> The dumbbell chart quantifies spatial centralization in a single view. In highly resilient democracies (Germany, UK), the gap between capital and peripheral intervention rates is minimal, that means violence is "policing-oriented." In backsliding regimes (Slovakia, Czech Republic), the line segments stretch dramatically, with capital city rates far exceeding peripheral rates, indicating the state has abandoned broad territorial policing in favor of <strong>concentrated, highly visible deterrence</strong> in core political hubs.
                    </div>
                </div>
            </div>
        </div>
        <div class="col-lg-6">
            <div class="main-card">
                <div class="main-card-body">
                    <h4 class="sub-section-title">Plot 5: Annual Evolution of the Capital–Periphery Intervention Gap</h4>
                    <div class="plot-container">
                        {plot5_html}
                    </div>
                    <div class="insight-box mt-3">
                        <strong>Response to RQ2:</strong> The trend lines track how the spatial gap evolves over time. In the "All Countries" view, the capital-periphery gap remains relatively stable for consolidated democracies but shows widening divergence for backsliding ones. Selecting individual countries via the dropdown reveals that Slovakia and Czech Republic exhibit growing spatial concentration of violence in specific years (particularly during crisis periods), providing <strong>temporal validation</strong> that spatial deterrence intensifies when institutions weaken.
                    </div>
                </div>
            </div>
        </div>
    </div>

    <hr class="divider">

    <!-- ================================================================== -->
    <!-- SECTION 5: RQ3 — Crisis Amplifier (Plot 6 & Plot 7)                -->
    <!-- ================================================================== -->
    <div class="main-card" style="border-left: 4px solid #c0392b;">
        <div class="main-card-body">
            <div class="d-flex align-items-center mb-2">
                <span class="rq-badge rq3-badge" style="font-size:1rem; padding:6px 18px;">RQ3</span>
                <h3 class="section-title mb-0" style="border-bottom:none; margin-bottom:0;">Crisis Amplifier: Exogenous Shocks as Institutional Stress Tests</h3>
            </div>
            <p class="method-text mb-0">
                Do external crises (2020_Covid-19, 2022_Russia-Ukraine War, 2023_European Energy / Oil Crisis) act as pressure tests that expose and accelerate hidden institutional fragility? These two dynamic visualizations track how crisis events produce disproportionate violence amplification in backsliding regimes and potentially reset state violence baselines permanently.
            </p>
        </div>
    </div>

    <!-- Plot 6: Full Width -->
    <div class="main-card">
        <div class="main-card-body">
            <h4 class="sub-section-title">Plot 6: Dynamic Pressure Test — Deviation from Normal Baseline</h4>
            <div class="plot-container">
                {plot6_html}
            </div>
            <div class="insight-box mt-3">
                <strong>Response to RQ3:</strong> By pressing play, we track each country's deviation from a "normal baseline" (calculated from non-crisis years). Points adhering to the 45° dashed line indicate the crisis had no disproportionate effect, that can see the state (or the citizens) responded proportionally. Points breaking violently <em>above</em> the line (especially visible during 2020-Q1 for Czech Republic) represent <strong>"Disproportionate Violence Amplification"</strong>, proving that crisis uncovers hidden institutional fragility. The hover reveals V-Dem indices alongside crisis deviations, allowing direct correlation between democratic quality and crisis vulnerability.
            </div>
        </div>
    </div>

    <!-- Plot 7: Full Width -->
    <div class="main-card">
        <div class="main-card-body">
            <h4 class="sub-section-title">Plot 7: Spatial-Temporal Stress Matrix — Monthly Deviations Heatmap</h4>
            <div class="plot-container">
                {plot7_html}
            </div>
            <div class="insight-box mt-3">
                <strong>Response to RQ3:</strong> This heatmap tracks monthly deviations from country-specific baselines. Red blocks indicate months where state violence exceeded normal parameters; blue indicates below-baseline. Playing the animation reveals that crises do not cause uniform annual increases, rather, they cause <strong>intense, short-duration "spikes"</strong> (e.g., 2020 spring for Czech Republic). Crucially, by 2023 (energy crisis), the same countries show less red, that not because they became less violent, but because the <em>baseline itself was permanently raised</em> by the 2020 shock. This proves the <strong>"Authoritarian Inertia" effect</strong>: democratic backsliding produces irreversible shifts in state violence that become the new normal.
            </div>
        </div>
    </div>

    <hr class="divider">

    <!-- ================================================================== -->
    <!-- SECTION 6: Summary of Findings & Research Question Responses        -->
    <!-- ================================================================== -->
    <div class="main-card mb-4">
        <div class="main-card-body">
            <h3 class="section-title">Summary of Findings &amp; Research Question Responses</h3>
            <div class="row g-4">
                <div class="col-md-4">
                    <div class="summary-box">
                        <h5><span class="rq-badge rq1-badge">RQ1</span> The Non-Linear Threshold</h5>
                        <p>
                            The data robustly supports RQ1. Democratic backsliding does not gradually increase state violence. Instead, it erodes the institutional buffers (judicial oversight, police professionalism) until a critical threshold is crossed, at which point state force is deployed disproportionately. The trajectory plots confirm that resilient democracies maintain a "flatline" of restraint, while backsliding ones exhibit "cardiogram-like" pulses — sudden, high-amplitude spikes that betray the collapse of institutional self-restraint.
                        </p>
                    </div>
                </div>
                <div class="col-md-4">
                    <div class="summary-box">
                        <h5><span class="rq-badge rq2-badge">RQ2</span> Spatial Centralization of Violence</h5>
                        <p>
                            In response to RQ2, the plots showcase that declining democracies fundamentally alter their <em>geography of repression</em>. Rather than dispersing force evenly across the territory (as healthy democracies do for public safety), they concentrate violence in capital cities to maximize media deterrence and political signaling. The dumbbell chart and spatial maps reveal a widening "Violence Gap" between political centers and peripheries — effectively documenting the state's withdrawal from broad administrative service in favor of <strong>precision deterrence</strong>.
                        </p>
                    </div>
                </div>
                <div class="col-md-4">
                    <div class="summary-box">
                        <h5><span class="rq-badge rq3-badge">RQ3</span> Crisis as Catalyst &amp; Irreversible Baseline Shift</h5>
                        <p>
                            RQ3 is confirmed: exogenous crises act as brutal stress tests. The dynamic scatter plots and heatmaps show that while stable democracies absorb shocks without breaking baseline norms, backsliding regimes experience structural fractures during crises. Most critically, the heatmap reveals an <strong>"Authoritarian Inertia" effect</strong>: the 2020 Covid-19 shock permanently raised violence baselines in fragile democracies, so that by 2023, new crises no longer produced visible red spikes — not because violence decreased, but because the <em>new normal was already repressive</em>. This proves democratic backsliding produces irreversible shifts in state coercion.
                        </p>
                    </div>
                </div>
            </div>

            <div class="row mt-4">
                <div class="col-12">
                    <div class="insight-box" style="border-left: 4px solid #718096; background-color: #f9fafb;">
                        <h5 style="color: #2d3748; font-weight: 600; margin-bottom: 8px;">Potential Methodological Note &amp; Q&amp;A Defense</h5>
                        <p style="font-size: 0.9rem; margin-bottom: 8px;">
                            <strong>Why LOWESS and visual inference instead of regression?</strong> Panel data with autocorrelation and temporal granularity mismatch invites endogeneity crises. LOWESS non-parametric estimation and cross-national time-series trajectories provide rigorous exploratory evidence of non-linear thresholds without violating statistical assumptions.
                        </p>
                        <p style="font-size: 0.9rem; margin-bottom: 8px;">
                            <strong>Why two-sample comparison instead of spatial econometrics?</strong> ACLED event-level spatial distributions are heavily influenced by media coverage density, making spatial regression models prone to measurement error. Capital vs. periphery comparisons using t-tests provide the most robust and parsimonious identification strategy.
                        </p>
                        <p style="font-size: 0.9rem; margin-bottom: 0;">
                            <strong>Why DID visual logic instead of formal DID regression?</strong> The quarterly panel's limited pre-treatment periods and the simultaneity of crises across countries constrain formal DID estimation. The visual DID framework (parallel pre-trend → structural divergence at T0) provides transparent, assumption-light causal evidence.
                        </p>
                    </div>
                </div>
            </div>

            <p class="mt-4" style="font-size: 0.85rem; color: #718096; line-height: 1.6;">
                <strong>AI Usage Acknowledgment:</strong> AI coding assistants were utilized strictly for code debugging, syntax structuring, and visual layout formatting. The specific topic choice, variable selection, methodological design, and all analytical insights represent entirely independent student work.
            </p>
        </div>
    </div>

    <hr class="divider">

    <!-- ================================================================== -->
    <!-- SECTION 7: Analytical Report                                       -->
    <!-- ================================================================== -->
    <div class="main-card" style="border-left: 4px solid #8e44ad;">
        <div class="main-card-body">
            <h3 class="method-title" style="border-bottom-color: #8e44ad;">
                Analytical Report: Democracy, Protest, and State Violence in Europe
            </h3>

            <!-- 7.1 Introduction & Problem Background -->
            <h4 class="sub-section-title">Introduction and Background</h4>
            <div class="method-text report-body">
                <p>Over the past decade, European democracies have confronted an unprecedented convergence of exogenous shocks and endogenous institutional erosion. The Covid-19 pandemic, the Russia-Ukraine war, and the rise of populist-authoritarian movements have collectively tested the resilience of democratic governance across the continent. While much of the existing scholarship on state repression focuses on non-democratic or hybrid regimes (Davenport, 2007), comparatively less attention has been directed toward understanding how ostensibly stable democracies manage dissent during periods of institutional stress. This gap is particularly consequential because democratic backsliding in established democracies tends to be incremental and legally cloaked, making it harder to detect through conventional regime-classification metrics (Bermeo, 2016).</p>

                <p>This dashboard investigates a critical yet under-examined dimension of democratic backsliding: the state's willingness to deploy coercive force against its own citizens. Rather than relying on absolute fatality counts or binary regime-type indicators, we construct a <em>State Intervention Rate</em> (the ratio of protests interrupted by police or security forces to total protest events) as a continuous dependent variable. This approach captures the <em>propensity</em> for state violence independent of protest volume, thereby isolating the state's behavioral calculus from the scale of citizen mobilization. By merging micro-level event data from the Armed Conflict Location and Event Data Project (ACLED) with macro-level institutional indices from the Varieties of Democracy (V-Dem) project, we are able to examine how fluctuations in democratic quality drive changes in the state's tactical response to dissent across five European countries: Germany, France, the United Kingdom, the Czech Republic, and Slovakia.</p>

                <p>The choice of these five countries is analytically deliberate. Germany, France, and the United Kingdom represent mature Western European democracies with robust institutional checks, while the Czech Republic and Slovakia serve as post-communist cases where democratic consolidation is comparatively more fragile. This West-East gradient enables us to probe whether democratic quality operates as a continuous moderator of state violence or whether a categorical \"democratic shield\" effect prevails. The temporal scope (2018-2024) further ensures coverage of both pre-crisis and crisis periods, facilitating quasi-natural experiments that illuminate the crisis amplification hypothesis.</p>
            </div>

            <!-- 7.2 RQ1: Threshold Effect -->
            <h4 class="sub-section-title">RQ1 &mdash; The Threshold Effect: Democratic Erosion and Non-Linear Violence</h4>
            <div class="method-text report-body">
                <p>Research Question 1 asks whether democratic decline increases state violence in a linear fashion or whether a critical threshold exists beyond which institutional restraints collapse. The theoretical foundation for this inquiry draws on Davenport's (2007) state repression framework, which posits that the relationship between regime type and repressive behavior is mediated by institutional constraints such as judicial independence, legislative oversight, and media freedom. However, more recent work on democratic backsliding suggests that these constraints may erode gradually until a tipping point is reached, at which point the rate of state violence accelerates non-linearly (Luhrmann & Lindberg, 2019).</p>

                <p>Plot 1, the Spatial Distribution of Protests and State Intervention Hotspots, provides the first layer of evidence. By mapping protest events across five countries with intervention levels color-coded (blue for no intervention, amber for low intervention, red for high intervention), the visualization reveals a striking geographic concentration of state violence in capital cities. France's Paris, the United Kingdom's London, and Germany's Berlin consistently register the highest number of high-intervention events, even though these cities also host the largest absolute volumes of protest. This pattern suggests that the state's coercive response is not merely a function of protest density but is also shaped by the political significance of the location&mdash;a finding consistent with Earl et al.'s (2003) argument that authorities escalate repression where the perceived political cost of tolerating dissent is highest.</p>

                <p>Plot 2, the Cross-National Repression Rate Trajectories, offers the temporal dimension needed to evaluate the threshold hypothesis. The line chart tracks monthly repression rates across all five countries from 2020 to 2025. The most notable finding is the sharp divergence during the Covid-19 pandemic (2020-2021), when France and Slovakia exhibited pronounced spikes in their intervention rates while Germany maintained relative stability. This divergence is particularly telling because all three countries faced comparable public health threats and implemented similar lockdown measures. The differential state response, therefore, cannot be attributed solely to crisis severity and must instead reflect underlying differences in institutional quality and political culture (Graham-Harrison et al., 2020). The peak annotations in the individual country views reveal that France's maximum repression rate significantly exceeded the five-country average, while Germany's peak remained below it&mdash;supporting the hypothesis that higher V-Dem Liberal Democracy Index scores (v2x_libdem) serve as a buffer against disproportionate state violence during crises.</p>
            </div>

            <!-- 7.3 RQ2: Spatial Deterrence -->
            <h4 class="sub-section-title">RQ2 &mdash; Spatial Deterrence: Capital-Centric Coercion and the Geography of State Violence</h4>
            <div class="method-text report-body">
                <p>Research Question 2 examines how institutional decay alters the spatial distribution of state violence, asking whether backsliding regimes concentrate coercive force in political centers to maximize deterrence. The theoretical logic follows from Tilly's (2003) observation that state formation historically involves the concentration of coercive capacity in strategic locations, and from recent work on the spatial politics of repression which shows that security forces disproportionately target protests in politically symbolically significant areas (Weidmann & Ward, 2010).</p>

                <p>Plot 3, the Cross-border Space Deterrence Comparison Map, animates annual protest and intervention patterns across all five countries. The animation reveals that state interventions (red dots) cluster disproportionately in capital cities, particularly Paris, London, and Prague, even when accounting for the higher baseline volume of protests in these cities. The West-East comparison toggle further illuminates this pattern: Western European capitals exhibit more dispersed intervention patterns, with peripheral cities like Lyon, Manchester, and Munich also recording state interventions, whereas Eastern European countries show a more pronounced capital-centric concentration&mdash;suggesting that in less consolidated democracies, coercive resources are more narrowly concentrated in the political center.</p>

                <p>Plots 4 and 5 provide the quantitative backbone for this analysis. Plot 4, the Capital-Local Violence Gap dumbbell chart, directly compares intervention rates between capital cities and other regions. The visualization demonstrates that all five countries exhibit a positive gap, meaning that state intervention rates are systematically higher in capital cities than in the rest of the country. However, the magnitude of this gap varies significantly across countries: France shows one of the widest gaps, consistent with its relatively higher baseline repression rate and the political centrality of Paris as the site of the Gilets Jaunes protests and subsequent police crackdowns. Slovakia, by contrast, shows a narrower gap, which may reflect either more geographically dispersed protest activity or a more uniform deployment of coercive capacity.</p>

                <p>Plot 5, the Annual Gap Trend Chart, traces the evolution of this capital-periphery gap over time. The critical finding is that the gap widened during the Covid-19 pandemic (2020-2021) and again during the early phases of the Russia-Ukraine war (2022), supporting the argument that exogenous crises amplify the state's tendency to concentrate coercive force in politically sensitive locations. This pattern is consistent with what Gorski (2003) terms the \"disciplinary revolution\" logic: during periods of perceived existential threat, states redirect coercive resources toward the locations where the political stakes of dissent are highest.</p>
            </div>

            <!-- 7.4 RQ3: Crisis Amplifier -->
            <h4 class="sub-section-title">RQ3 &mdash; Crisis as Amplifier: Exogenous Shocks and Institutional Stress Testing</h4>
            <div class="method-text report-body">
                <p>Research Question 3 investigates whether exogenous crises act as pressure tests that expose and accelerate underlying institutional fragility. The crisis amplification hypothesis draws on Pierson's (2004) work on institutional vulnerability during periods of rapid change, arguing that crises simultaneously increase the demand for state coercive action and weaken the institutional constraints that normally restrain such action. In the European context, the Covid-19 pandemic and the Russia-Ukraine war provide two quasi-natural experiments for testing this hypothesis.</p>

                <p>Plot 6, the Dynamic Pressure Test Scatter Plot, is the most direct visualization of this hypothesis. The scatter plot positions each country-quarter observation relative to a 45-degree baseline, where the x-axis represents the normal (peacetime) baseline intervention rate and the y-axis represents the current quarter's intervention rate. Points falling above the 45-degree line indicate disproportionate violence amplification during crises, while points on or below the line indicate proportional or restrained responses. The animated timeline reveals that during Q2-Q3 2020 (the peak of Covid-19 lockdown enforcement), France's observation point moved significantly above the line, indicating that its crisis-period repression rate exceeded what would be predicted from its baseline&mdash;a pattern consistent with the disproportionate violence amplification predicted by the crisis hypothesis. Germany, by contrast, remained closer to the line, suggesting that its stronger institutional constraints (higher V-Dem scores on rule of law and liberal democracy) acted as a brake on crisis-driven escalations.</p>

                <p>Plot 7, the Spatial-Temporal Stress Matrix Heatmap, complements the scatter plot by decomposing deviations from baseline into monthly increments for each country. The heatmap reveals concentrated \"stress bands\" corresponding to crisis periods: a deep red band in early-to-mid 2020 across all countries (Covid-19), followed by a secondary stress band in late 2022 and early 2023 (Russia-Ukraine war's energy crisis and inflation shock). Notably, the 2020 stress band is more intense and uniform across countries than the 2022-2023 band, suggesting that the pandemic represented a more systemic shock to democratic governance than the geopolitical crisis&mdash;a finding consistent with Maerz et al.'s (2020) observation that pandemic-related emergency measures frequently bypassed standard legislative oversight, creating opportunities for executive overreach.</p>
            </div>

            <!-- 7.5 Cross-National Comparative Insights -->
            <h4 class="sub-section-title">Cross-National Comparative Insights and Implications</h4>
            <div class="method-text report-body">
                <p>The cross-national comparison across the seven visualizations yields several analytically significant patterns. First, there is a clear democratic quality gradient in state repressive behavior: countries with higher V-Dem Liberal Democracy Index and Rule of Law scores (Germany, United Kingdom) consistently exhibit lower baseline repression rates and smaller crisis-driven spikes than countries with lower scores (France, Slovakia). This finding supports the argument that democratic institutions function as genuine constraints on state violence rather than mere epiphenomena of regime type (Davenport, 2007). Second, the spatial analysis reveals that even in high-quality democracies, the capital-city premium in state violence persists, suggesting that this pattern reflects structural features of governance rather than democratic deficits per se. Third, the crisis amplification effect is universal but unevenly distributed: all five countries experienced elevated repression during crises, but the magnitude of the response was modulated by institutional quality. This finding has normative implications for democratic governance, suggesting that investments in institutional resilience (judicial independence, legislative oversight, press freedom) yield measurable dividends in terms of reduced state violence during crises (Luhrmann & Lindberg, 2019).</p>
            </div>

            <hr class="divider">

            <!-- 7.6 Technology Overview -->
            <h4 class="sub-section-title">Technology Overview</h4>
            <div class="method-text report-body">
                <p>This interactive HTML dashboard was constructed using a Python-based computational pipeline that integrates data science, statistical visualization, and web technologies for the purpose of political analysis. The core data processing and transformation were performed using <strong>Pandas</strong> and <strong>NumPy</strong>, which enabled efficient manipulation of the ACLED micro-event dataset and V-Dem macro-institutional panel data. All seven analytical visualizations were generated using <strong>Plotly.js</strong> (via the <strong>plotly.graph_objects</strong> Python API), which provides interactive features including hover tooltips, animated time-series sliders, country-specific toggling, and dynamic frame-based animations&mdash;capabilities that are essential for exploratory political data analysis where users need to examine temporal dynamics at granular levels.</p>

                <p>The choice of these technologies is analytically motivated. In political science research, static visualizations (e.g., bar charts, scatter plots) often fail to capture the temporal complexity and geographic heterogeneity of protest and repression dynamics. Plotly's interactive framework addresses this limitation by enabling users to animate protest trajectories over time, filter by country or region, and drill down into specific time periods&mdash;all within a single, self-contained HTML file. The dashboard layout was built with <strong>Bootstrap 5.3</strong> for responsive grid design and custom CSS for consistent visual styling. The Mapbox integration in Plots 1 and 3 leverages CartoDB's Positron base map tiles to provide geographic context without visual clutter, which is particularly important when overlaying hundreds of protest location markers with varying sizes and colors. The entire pipeline&mdash;from data ingestion to HTML export&mdash;runs within a Jupyter Notebook, ensuring reproducibility and transparency. The output is a standalone HTML file that requires no server-side infrastructure, making it ideal for academic presentations, peer review, and archival distribution.</p>
            </div>

            <hr class="divider">

            <!-- 7.7 References (APA 7th Edition) -->
            <h4 class="sub-section-title">References List</h4>
            <div class="method-text report-body" style="font-size: 0.9rem; line-height: 1.8;">
                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Bermeo, N. (2016). On democratic backsliding. <em>Journal of Democracy</em>, <em>27</em>(1), 5&ndash;19. <a href="https://doi.org/10.1353/jod.2016.0012" target="_blank">https://doi.org/10.1353/jod.2016.0012</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Davenport, C. (2007). State repression and political order. <em>Annual Review of Political Science</em>, <em>10</em>, 1&ndash;23. <a href="https://doi.org/10.1146/annurev.polisci.10.101405.143216" target="_blank">https://doi.org/10.1146/annurev.polisci.10.101405.143216</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Earl, J., Soule, S. A., & McCarthy, J. D. (2003). Protest under fire? Explaining the policing of protest. <em>American Sociological Review</em>, <em>68</em>(4), 581&ndash;606. <a href="https://doi.org/10.2307/1519740" target="_blank">https://doi.org/10.2307/1519740</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Gorski, P. S. (2003). <em>The disciplinary revolution: Calvinism and the rise of the state in early modern Europe</em>. University of Chicago Press</em>. <a href="https://doi.org/10.7208/9780226304861" target="_blank">https://doi.org/10.7208/9780226304861</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Graham-Harrison, E., Giuffrida, A., Smith, H., & Tondo, L. (2020, April 3). Europe's police getting heavy-handed during coronavirus lockdowns. <em>The Guardian</em>.

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Luhrmann, A., & Lindberg, S. I. (2019). A third wave of autocratization is here: What is new about it? <em>Democratization</em>, <em>26</em>(7), 1095&ndash;1113. <a href="https://doi.org/10.1080/13510347.2019.1582029" target="_blank">https://doi.org/10.1080/13510347.2019.1582029</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Maerz, S. F., Luhrmann, A., Hellmeier, S., Edgell, A. B., & Lindberg, S. I. (2020). Autocratization surges &ndash; Resistance grows: Democracy report 2020. <em>V-Dem Institute Working Paper</em>, 2020:75. <a href="https://www.v-dem.net/documents/14/dr_2020_dqumD5e.pdf</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Pierson, P. (2004). <em>Politics in time: History, institutions, and social analysis</em>. Princeton University Press</em>. <a href="https://doi.org/10.1515/9781400841080" target="_blank">https://doi.org/10.1515/9781400841080</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Tilly, C. (2003). <em>The politics of collective violence</em>. Cambridge University Press</em>. <a href="https://rodrigomorenog.wordpress.com/wp-content/uploads/2021/08/tilly-the-politics-of-collective-violence-cambridge-university-press-2003.pdf" target="_blank">https://rodrigomorenog.wordpress.com/wp-content/uploads/2021/08/tilly-the-politics-of-collective-violence-cambridge-university-press-2003.pdf</a></p>

                <p style="padding-left: 2em; text-indent: -2em; margin-bottom: 10px;">Weidmann, N. B., & Ward, M. D. (2010). Predicting conflict in space and time. <em>Journal of Conflict Resolution</em>, <em>54</em>(6), 883&ndash;901. <a href="https://doi.org/10.1177/0022002711405502" target="_blank">https://doi.org/10.1177/0022002711405502</a></p>
            </div>
        </div>
    </div>

    <div class="footer-text">
        Plan B Dashboard — POLI 3148 Data Science in Politics and Public Administration — 2025
    </div>

</div>

</body>
</html>"""

# =============================================================================
# Write the integrated dashboard HTML to file
# =============================================================================
output_filename = "Final_Integrated_Dashboard.html"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print(f"\nIntegrated dashboard generated: {output_filename}")


Generating Plot 1...
Generating Plot 2...
Generating Plot 3...
Generating Plot 4...
Generating Plot 5...
Generating Plot 6...
Generating Plot 7...

Assembling integrated dashboard HTML...

Integrated dashboard generated: Final_Integrated_Dashboard.html
